#  RAG 체인 구성 - Naïve RAG 구현

### **학습 목표:**
1. Retriever의 개념과 역할을 이해한다
2. 벡터 저장소에서 다양한 검색 방법(Top-K, Threshold, MMR)을 활용할 수 있다
3. LangChain을 사용하여 RAG 파이프라인을 구성할 수 있다
4. Gradio를 활용한 스트리밍 RAG 챗봇을 구현할 수 있다

### **실습 자료**: 
- data/transformer.pdf

---

# 환경 설정 및 준비

- 필수 라이브러리: langchain-chroma, langchain-community, faiss-cpu, langchain-openai, gradio
- 환경변수: OPENAI_API_KEY 설정 필요


`(1) Env 환경변수`

In [1]:
import os
import warnings

# Tokenizers 병렬 처리 경고 억제
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 경고 억제 (선택사항)
warnings.filterwarnings('ignore', category=UserWarning)

from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [ ]:
import os
from glob import glob

`(3) 문서 로드`

In [2]:
from langchain_community.document_loaders import PyPDFLoader

# PDF 로더 초기화
pdf_loader = PyPDFLoader('./data/transformer.pdf')

# 동기 로딩
pdf_docs = pdf_loader.load()
print(f'PDF 문서 개수: {len(pdf_docs)}')

/Users/mykim/education/001_chatbot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PDF 문서 개수: 15


`(4) 텍스트 분할`

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

# Hugging Face의 임베딩 모델 생성
embeddings_huggingface = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

# 토크나이저 직접 접근
tokenizer = embeddings_huggingface._client.tokenizer

# 토크나이저를 사용한 예시
text = """연구 목적: 공공 서비스 디자인 실무 맥락에서 공무원의 요구를 반영해 설계된 AI 기반 합성 사용자의 활용 가능성을 탐구한다.

데이터 규모: 아직 확실하게 어떤 분야의 데이터를 사용할지 정하지는 않았습니다. 다만 '공공 데이터 포털'에 오픈되어 있는 데이터를 사용할 예정입니다. 그래서 아마도 사용자 1유형당 5-8개 항목이 사용될것 같습니다.

페르소나 수: 동일한 사용자 유형을 기준으로 한 페르소나 1–2명

시뮬레이션 방식: 텍스트로 gpt처럼 채팅 방식을 생각하고 있습니다."""

tokens = tokenizer(text)
print(tokens)

# 숫자로 바뀐 토큰을 다시 글자로 복구해서 확인하기
decoded_text = tokenizer.convert_ids_to_tokens(tokens['input_ids'])
print(decoded_text)

# 토크나이저 설정 확인
print(tokenizer.model_max_length)  # 최대 토큰 길이
print(tokenizer.vocab_size)        # 어휘 크기

{'input_ids': [0, 27568, 99740, 12, 133138, 17403, 55396, 16907, 7286, 98720, 29961, 1180, 6, 126031, 367, 70814, 688, 158158, 1963, 136435, 5191, 38730, 96685, 51638, 3276, 77812, 367, 48284, 175164, 413, 123575, 4584, 9514, 5, 74168, 143879, 12, 79795, 6, 191093, 5681, 26013, 66964, 367, 207580, 82934, 1190, 7063, 11289, 769, 82476, 16367, 5, 127859, 242, 7849, 7849, 74168, 6, 233990, 25, 480, 109662, 21988, 2901, 207580, 82934, 84666, 5826, 5, 45229, 96897, 1048, 77812, 106, 6943, 8333, 7641, 6, 138573, 6027, 176200, 469, 10993, 24788, 32657, 96040, 5, 82955, 8495, 3999, 3497, 1020, 12, 8354, 1599, 993, 77812, 167596, 413, 200765, 3103, 82955, 8495, 3999, 3497, 106, 1104, 304, 5576, 6452, 243716, 224199, 116183, 12, 239355, 1083, 706, 6328, 29882, 27235, 33994, 116183, 413, 19243, 2382, 3292, 5, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [7]:
# 토큰 수를 계산하는 함수
def count_tokens(text):
    return len(tokenizer(text)['input_ids'])

# 토큰 수 계산
text = """연구 목적: 공공 서비스 디자인 실무 맥락에서 공무원의 요구를 반영해 설계된 AI 기반 합성 사용자의 활용 가능성을 탐구한다.

데이터 규모: 아직 확실하게 어떤 분야의 데이터를 사용할지 정하지는 않았습니다. 다만 '공공 데이터 포털'에 오픈되어 있는 데이터를 사용할 예정입니다. 그래서 아마도 사용자 1유형당 5-8개 항목이 사용될것 같습니다.

페르소나 수: 동일한 사용자 유형을 기준으로 한 페르소나 1–2명

시뮬레이션 방식: 텍스트로 gpt처럼 채팅 방식을 생각하고 있습니다."""
print(count_tokens(text))

129


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할기 생성
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,                      
    chunk_overlap=100,           
    length_function=count_tokens,         # 토큰 수를 기준으로 분할
    separators=["\n\n", "\n",],   # 구분자 - 재귀적으로 순차적으로 적용 
)

# 텍스트 분할
chunks = text_splitter.split_documents(pdf_docs)
print(f"생성된 텍스트 청크 수: {len(chunks)}")
print(f"각 청크의 길이: {list(len(chunk.page_content) for chunk in chunks)}")
print(f"각 청크의 토큰 수: {list(count_tokens(chunk.page_content) for chunk in chunks)}")

생성된 텍스트 청크 수: 38
각 청크의 길이: [1379, 1796, 1831, 1857, 1293, 1610, 503, 1557, 1278, 1370, 1611, 835, 1416, 1679, 999, 1765, 1605, 540, 1220, 1647, 927, 1213, 1689, 717, 1409, 1626, 624, 1411, 1438, 914, 1496, 1340, 847, 812, 470, 438, 470, 441]
각 청크의 토큰 수: [337, 415, 405, 419, 327, 424, 127, 389, 294, 383, 414, 206, 419, 417, 226, 418, 395, 149, 393, 405, 223, 356, 411, 181, 394, 405, 188, 424, 400, 278, 423, 413, 252, 190, 131, 117, 131, 113]


In [10]:
# 청크의 텍스트 확인
print(chunks[1].page_content)

be superior in quality while being more parallelizable and requiring significantly
less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-
to-German translation task, improving over the existing best results, including
ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task,
our model establishes a new single-model state-of-the-art BLEU score of 41.8 after
training for 3.5 days on eight GPUs, a small fraction of the training costs of the
best models from the literature. We show that the Transformer generalizes well to
other tasks by applying it successfully to English constituency parsing both with
large and limited training data.
∗Equal contribution. Listing order is random. Jakob proposed replacing RNNs with self-attention and started
the effort to evaluate this idea. Ashish, with Illia, designed and implemented the first Transformer models and
has been crucially involved in every aspect of this work. Noam proposed scaled dot-product attention,

# 벡터 저장소 기반 RAG 검색기 (Retriever)



`(1) 벡터 저장소 초기화`
- chroma 사용
- cosine distance 기준으로 인덱싱 

In [ ]:
from langchain_chroma import Chroma

# Chroma 벡터 저장소 생성하기
chroma_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_huggingface,    # huggingface 임베딩 사용
    collection_name="db_transformer_cosine",    # 컬렉션 이름
    persist_directory="./chroma_db",
    collection_metadata = {'hnsw:space': 'cosine'}, # l2, ip, cosine 중에서 선택 
)

# 현재 저장된 컬렉션 데이터 확인
chroma_db.get()

In [ ]:
chroma_db._collection.count()

`(2) Top K`

In [ ]:
chroma_k_retriever = chroma_db.as_retriever(
    search_kwargs={"k": 2},
)

query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
retrieved_docs = chroma_k_retriever.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"-{i}-\n{doc.page_content[:100]}...{doc.page_content[-100:]} [출처: {doc.metadata['source']}]")
    print("-" * 100)

`(3) 임계값 지정`
- Similarity score threshold (기준 스코어 이상인 문서를 대상으로 추출)

In [ ]:
from langchain_community.utils.math import cosine_similarity

chroma_threshold_retriever = chroma_db.as_retriever(
    search_type='similarity_score_threshold',       # cosine 유사도
    search_kwargs={'score_threshold': 0.1, 'k':5},  # 0.5 이상인 문서를 추출
)

query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
retrieved_docs = chroma_threshold_retriever.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    score = cosine_similarity(
        [embeddings_huggingface.embed_query(query)], 
        [embeddings_huggingface.embed_query(doc.page_content)]
        )[0][0]
    print(f"-{i}-\n{doc.page_content[:100]}...{doc.page_content[-100:]} [유사도: {score}]")
    print("-" * 100)

`(4) MMR(Maximal Marginal Relevance) 검색`

In [ ]:
# MMR - 다양성 고려
chroma_mmr = chroma_db.as_retriever(
    search_type='mmr',
    search_kwargs={
        'k': 3,                 # 최종적으로 반환할 문서의 수
        'fetch_k': 8,           # 유사도 기준으로 먼저 가져올 후보 문서 수 (fetch_k >= k 권장)
        'lambda_mult': 0.5,     # 유사도와 다양성의 균형 (0=최대 다양성, 1=최대 유사도, 기본값=0.5)
        # lambda_mult가 낮을수록 서로 다른 내용의 문서를, 높을수록 쿼리와 유사한 문서를 우선 선택
        },
)


query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
retrieved_docs = chroma_mmr.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    score = cosine_similarity(
        [embeddings_huggingface.embed_query(query)], 
        [embeddings_huggingface.embed_query(doc.page_content)]
        )[0][0]
    print(f"-{i}-\n{doc.page_content[:100]}...{doc.page_content[-100:]} [유사도: {score}]")
    print("-" * 100)

`(5) metadata 필터링 검색`

In [ ]:
# 메타데이터 확인
chunks[0].metadata

In [ ]:
# 문서 객체의 metadata를 이용한 필터링
chrom_metadata = chroma_db.as_retriever(
    search_kwargs={
        'filter': {'source': './data/transformer.pdf'},
        'k': 5, 
        }
)

query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
retrieved_docs = chrom_metadata.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"-{i}-\n{doc.page_content} [출처: {doc.metadata['source']}]")
    print("-" * 100)

`(6) page_content 본문 필터링 검색`

In [ ]:
# page_content를 이용한 필터링
chroma_content = chroma_db.as_retriever(
    search_kwargs={
        'k': 2,
        'where_document': {'$contains': 'recurrent'},
        }
)

query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
retrieved_docs = chroma_content.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"-{i}-\n{doc.page_content} [출처: {doc.metadata['source']}]")
    print("-" * 100)

# [실습 프로젝트] Naive RAG 구현 

- 각 단계별 지시사항에 따라 코드를 완성하세요. 
- 제시된 지시사항과 LangChain 문서를 참조하여 시스템을 구성합니다. 

In [ ]:
# 3단계: 문서 관리

# 새로운 문서 추가
new_doc = Document(
    page_content="쿠버네티스 클러스터 운영 가이드",
    metadata={"type": "tutorial", "author": "김미영"}
)
new_id = str(uuid.uuid4())
practice_db.add_documents(documents=[new_doc], ids=[new_id])
print(f"새 문서 추가 완료: {new_id}")

# 특정 문서 삭제 (첫 번째 문서 삭제)
delete_id = doc_ids[0]
practice_db.delete(ids=[delete_id])
print(f"문서 삭제 완료: {delete_id}")

`(1) 벡터 저장소 설정`
- HuggingFace에서 지원하는 BAAI/bge-m3 임베딩 모델을 사용하여 문서를 벡터화
- FAISS DB를 벡터 스토어로 사용 (IndexFlatL2 사용: 유클리드 거리)

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings  

# Hugging Face의 임베딩 모델 생성
# 힌트: HuggingFaceEmbeddings(model_name="BAAI/bge-m3") 사용
embeddings_model = None

# 임베딩 차원 확인
embedding = embeddings_model.embed_query("test")
print(f"임베딩 차원: {len(embedding)}")

In [ ]:
# Ollama 임베딩 모델을 사용한 FAISS 벡터 저장소 생성
import faiss 
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

# FAISS 인덱스 초기화 (유클리드 거리 사용)
dim = 1024  # 임베딩 차원
faiss_index = faiss.IndexFlatL2(dim)  

# FAISS 벡터 저장소 생성
faiss_db = None

# 저장된 문서의 갯수 확인
print(faiss_db.index.ntotal)

In [ ]:
import uuid

# 문서 id 생성
doc_ids = [str(uuid.uuid4()) for _ in range(len(chunks))]

# 문서를 벡터 저장소에 저장
# 힌트: faiss_db.add_documents(chunks, ids=doc_ids) 사용
added_doc_ids = None

# 벡터 저장소에 저장된 문서를 확인
print(f"{len(added_doc_ids)}개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.")
print(added_doc_ids)

`(2) 검색기 정의`
- mmr 검색으로 상위 3개 문서 검색하는 Retriever 사용
- 다양성을 높이는 설정을 사용 

In [ ]:
# mmr 검색기 생성
# 힌트: faiss_db.as_retriever(search_type='mmr', search_kwargs={'k': 3, 'fetch_k': 10, 'lambda_mult': 0.3})
# lambda_mult를 낮게 설정하여 다양성을 높임
faiss_mmr_retriever = None

In [ ]:
# 검색 테스트 
query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
# 힌트: faiss_mmr_retriever.invoke(query) 사용
retrieved_docs = None

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"-{i}-\n{doc.page_content[:100]}...{doc.page_content[-100:]}")
    print("-" * 100)

`(3) RAG 프롬프트 구성`

- 작성 기준: 
    - LangChain의 ChatPromptTemplate 클래스 사용
    - 변수 처리는 {context}, {question} 형식 사용
    - 답변은 한글로 출력되도록 프롬프트 작성
    
- 아래 템플릿 코드를 기반으로 다음 내용을 참고하여 작성합니다. 

    1. 프롬프트 구성요소:
        - 작업 지침
        - 컨텍스트 영역
        - 질문 영역
        - 답변 형식 가이드

    2. 작업 지침:
        - 컨텍스트 기반 답변 원칙
        - 외부 지식 사용 제한
        - 불확실성 처리 방법
        - 답변 불가능한 경우의 처리 방법

    3. 답변 형식:
        - 핵심 답변 섹션
        - 근거 제시 섹션
        - 추가 설명 섹션 (필요시)

    4. 제약사항 반영:
        - 답변은 사실에 기반해야 함
        - 추측이나 가정을 최소화해야 함
        - 명확한 근거 제시가 필요함
        - 구조화된 형태로 작성되어야 함

In [ ]:
# Prompt 템플릿 (예시)
from langchain_core.prompts import ChatPromptTemplate

template = """Answer the question based only on the following context.

[Context]
{context}

[Question] 
{question}

[Answer]
"""

prompt = ChatPromptTemplate.from_template(template)

In [ ]:
# Prompt 템플릿 (여기에 작성하세요)
from langchain_core.prompts import ChatPromptTemplate

template = None

prompt = ChatPromptTemplate.from_template(template)

# 템플릿 출력
prompt.pretty_print()

`(4) RAG 체인 구성`
- LangChain의 LCEL 문법을 사용
- 검색 결과를 프롬프트의 'context'로 전달하고,
- 사용자가 입력한 질문을 그래도 프롬프트의 'question'에 전달
- LLM 설정:
    - ChatOpenAI 사용 ('gpt-4o-mini' 모델)
    - temperature: 답변의 일관성을 가져가는 설정값을 사용 
    - 기타 필요한 설정 
- 출력 파서: 문자열 부분만 출력되도록 구성

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# LLM 설정
# 힌트: ChatOpenAI(model='gpt-4o-mini', temperature=0) 사용
llm = None

# 문서 포맷팅
def format_docs(docs):
    return "\n\n".join([f"{doc.page_content}" for doc in docs])

# RAG 체인 생성
# 힌트: {'context': faiss_mmr_retriever | format_docs, 'question': RunnablePassthrough()} | prompt | llm | StrOutputParser()
rag_chain = None

# 체인 실행
query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
output = None

print(f"쿼리: {query}")
print("답변:")
print(output)

`(5) Gradio 스트리밍 구현`
- ChatInterface 사용
- `chain.stream()`으로 응답을 청크 단위로 스트리밍

In [ ]:
import gradio as gr
from typing import Iterator

# 스트리밍 응답 생성 함수
def get_streaming_response(message: str, history) -> Iterator[str]:
    
    # RAG Chain 실행 및 스트리밍 응답 생성
    response = ""
    for chunk in rag_chain.stream(message):
        if isinstance(chunk, str):
            response += chunk
            yield response

# Gradio 인터페이스 설정
# 힌트: gr.ChatInterface(fn=get_streaming_response, title="RAG 기반 질의응답 시스템", description="...", examples=[...])
demo = None

# 실행
demo.launch()

In [ ]:
# demo 실행 종료
demo.close()

## 제출용 실습코드

In [90]:
import os
import warnings
from dotenv import load_dotenv

# 1. API 키 로드: .env 파일에 작성된 OPENAI_API_KEY를 가져옵니다.
load_dotenv() 

# 2. 불필요한 경고 메시지는 무시하도록 설정합니다.
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings('ignore')

print("1단계: 준비물 챙기기 완료!")


1단계: 준비물 챙기기 완료!


In [91]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader

# 폴더 내의 모든 .txt 파일을 한꺼번에 가져옵니다. (garen.txt 포함)
loader = DirectoryLoader('./data/league_of_legends', glob="*.txt", loader_cls=TextLoader)
docs = loader.load()

print(f"2단계: {len(docs)}개의 파일을 성공적으로 읽어왔습니다.")


2단계: 29개의 파일을 성공적으로 읽어왔습니다.


In [92]:
from langchain_huggingface import HuggingFaceEmbeddings

# Hugging Face에서 지원하는 고성능 임베딩 모델을 생성합니다.
embeddings_model = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

# 토크나이저(단어 사전)를 준비하여 글자 수를 잴 준비를 합니다.
tokenizer = embeddings_model._client.tokenizer

print("3단계: 임베딩 모델(번역기) 세팅 완료!")


3단계: 임베딩 모델(번역기) 세팅 완료!


In [93]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. 토큰 수를 정확하게 재는 함수 정의
def count_tokens(text):
    return len(tokenizer(text)['input_ids'])

# 2. 텍스트 분할기(칼) 설정
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,          # 한 조각당 500토큰
    chunk_overlap=100,       # 100토큰은 겹치게 해서 맥락 유지
    length_function=count_tokens,
    separators=["\n\n", "\n", " ", ""]
)

# 3. 실제로 텍스트 조각내기
chunks = text_splitter.split_documents(docs)

print(f"4단계: 가렌 스토리를 총 {len(chunks)}개의 조각으로 칼질했습니다!")
print(f"--- 첫 번째 조각 미리보기 ---\n{chunks[0].page_content[:150]}...")


4단계: 가렌 스토리를 총 115개의 조각으로 칼질했습니다!
--- 첫 번째 조각 미리보기 ---
말자하
“우리는 시간을 초월한 존재다. 우리는 제물을 원한다.”

역할군
마법사

지역
공허

배경 이야기
어지러울 정도로 눈부신 슈리마의 태양 아래에는 언제나 예언의 힘이라는 축복을 받은 자들이 존재해 왔다. 그중 한 명인 말자하는 원래 나이 많은 행상인의 외동아들이...


In [ ]:
### 데이터 삭제 로직

#  메모리 초기화
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

dim = 1024 
faiss_index = faiss.IndexFlatL2(dim)

vectorstore = FAISS(
    embedding_function=embeddings_model,
    index=faiss_index,
    docstore=InMemoryDocstore({}),
    index_to_docstore_id={}
)

retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5, 'fetch_k': 10, 'lambda_mult': 0.7}
)

print("DB 초기화 완료! 이제 0개부터 다시 시작합니다.")


# DB 초기화 스크립트
import os
import glob

# 비우고 싶은 폴더 경로
db_path = "my_lol_db"

if os.path.exists(db_path):
    # 폴더 내부의 모든 파일 경로를 가져옵니다.
    files = glob.glob(os.path.join(db_path, "*"))
    
    for f in files:
        if os.path.isfile(f): # 파일인 경우에만 삭제
            os.remove(f)
    print(f"✨ '{db_path}' 폴더 내부를 깨끗이 비웠습니다!")
else:
    # 폴더가 없으면 새로 하나 만들어둡니다.
    os.makedirs(db_path)
    print(f"📁 '{db_path}' 폴더가 없어서 새로 생성했습니다.")

DB 초기화 완료! 이제 0개부터 다시 시작합니다.
✨ 'my_lol_db' 폴더 내부를 깨끗이 비웠습니다!


In [95]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

# 1. 규격 정하기
dim = 1024 
faiss_index = faiss.IndexFlatL2(dim)

# 2. 비어있는 Vector DB 만들기
vectorstore = FAISS(
    embedding_function=embeddings_model,
    index=faiss_index,
    docstore=InMemoryDocstore({}),
    index_to_docstore_id={}
)

# 3. 중요!! 먼저 지식 조각들(chunks)을 금고에 채웁니다.
vectorstore.add_documents(chunks)

# 4. 이제 내용물이 꽉 찬 금고를 파일로 저장합니다.
vectorstore.save_local("my_lol_db")

print(f"5단계 완료: {vectorstore.index.ntotal}개의 지식이 저장된 'my_lol_db' 폴더가 생성되었습니다!")

5단계 완료: 115개의 지식이 저장된 'my_lol_db' 폴더가 생성되었습니다!


In [ ]:
# MMR 방식: 단순히 비슷한 것만 찾는 게 아니라, 중복되지 않는 '다양한' 정보를 골라옵니다.
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={
        'k': 3,           # AI에게 최종적으로 전달할 지식 조각 개수
        'fetch_k': 10,    # 금고에서 후보를 10개 뽑은 뒤, 그 중 가장 알찬 3개를 골라냄
        'lambda_mult': 0.5 # 다양성 점수 (0.5가 가장 황금 밸런스!)
    }
)

# [테스트] 사서가 일을 잘하는지 확인해보기
test_query = "가렌은 어떤 무기를 사용해?"
search_results = retriever.invoke(test_query)

print(f"6단계 완료: 사서가 {len(search_results)}개의 관련 지식을 찾아왔습니다.")
print(f"--- 첫 번째 조각 내용 ---\n{search_results[0].page_content[:150]}...")

6단계 완료: 사서가 3개의 관련 지식을 찾아왔습니다.
--- 첫 번째 조각 내용 ---
그러면서 언젠가는 어떤 존재가 비교적 평화로운 이 시기에 종말을 가져올 것이라고 덧붙였다. 그 존재가 추방된 마법사일지, 심연에서 기어 나온 피조물일지, 아니면 상상을 초월하는 공포의 존재일지는 아직 알 수 없었다.

그 말을 증명이라도 하듯이, 숙부는 전투 중에 마법...


In [ ]:
### 기본 챗봇 구현

import gradio as gr
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser

# 1. 모델 설정
llm = ChatOpenAI(model_name="gpt-4.1-nano", temperature=0)

# 2. 정석적인 프롬프트 템플릿 정의 (MessagesPlaceholder 사용)
prompt = ChatPromptTemplate.from_messages([
    ("system", """너는 룬테라의 가이드야. 반드시 아래 규칙을 지켜서 답변해줘:
    [규칙]
    1. 제공된 [배경 지식] 정보를 바탕으로 친구에게 이야기하듯 친절한 반말로 답변해줘.
    2. 질문에 대한 정보가 [배경 지식]에 있다면 절대 숨기지 말고 상세히 알려줘.
    3. 만약 관련된 내용이 정말로 없다면 "미안해, 그 기록은 아직 우리 도서관에 없어."라고 답해줘."""),
    MessagesPlaceholder("chat_history"),
    ("system", "[배경 지식 정보]\n{context}"),
    ("human", "{user_input}")
])

# 3. 체인 생성
rag_chain = prompt | llm | StrOutputParser()

# 4. 채팅 처리 함수 (Streaming 적용)
def chat_with_bot(message, history):
    # (1) Gradio의 history를 LangChain 메시지 객체로 변환
    history_messages = []
    for msg in history:
        if msg['role'] == "user":
            history_messages.append(HumanMessage(content=msg['content']))
        elif msg['role'] == "assistant":
            history_messages.append(AIMessage(content=msg['content']))

    # (2) 사서(retriever)에게 관련 정보 찾아오기
    rel_docs = retriever.invoke(message)
    context_str = "\n".join([d.page_content for d in rel_docs])
    
    # (3) 답변 생성 및 스트리밍 (yield 사용)
    partial_message = ""
    for chunk in rag_chain.stream({
        "chat_history": history_messages,
        "context": context_str,
        "user_input": message
    }):
        partial_message += chunk
        yield partial_message

# 5. 디자인을 위한 HTML
# 텍스트 파일의 첫 줄(한글 이름) 추출하기
champion_names = []
for doc in docs:
    first_line = doc.page_content.split('\n')[0].strip()
    if first_line not in champion_names:
        champion_names.append(first_line)
champion_names.sort()
champions_str = ", ".join(champion_names)

description_html = f"""
<div style="background-color: #f0f4f8; padding: 20px; border-radius: 15px; border-left: 5px solid #2196F3; margin-bottom: 20px;">
    <h3 style="margin-top: 0; color: #1a237e;">📜 룬테라의 소리에 귀를 기울여보세요</h3>
    <p style="font-size: 1.1em; color: #34495e;">
        반가워요! 나는 룬테라의 이야기를 들려주는 가이드입니다. 😊<br>
        지금 내 지식 보관소에는 아래 챔피언들의 기록이 준비되어 있어요.
    </p>
    <div style="background: white; padding: 10px; border-radius: 8px; font-weight: bold; color: #2196F3; display: inline-block;">
        📍 보관된 기록: {champions_str}
    </div>
</div>
"""

# 6. Gradio 인터페이스 실행
demo = gr.ChatInterface(
    fn=chat_with_bot, 
    title="🌟 LoL 유니버스 도서관",
    description=description_html,
    examples=[[f"{champion_names[0]}에 대해 알려줘"], ["티모의 직업은 뭐야?"], ["데마시아에 대해 알려줘"]],
)

if __name__ == "__main__":
    demo.launch(share=True)    # share=True: 이 옵션을 넣으면 약 72시간 동안 유효한 외부 링크(예: `https://xxxx.gradio.live`)가 생김

* Running on local URL:  http://127.0.0.1:7890


python(44028) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


* Running on public URL: https://16cb188898fdacdbac.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [58]:
# 파일 추가 시
# 1. 기존 디비 불러오기
vectorstore = FAISS.load_local("my_lol_db", embeddings_model, allow_dangerous_deserialization=True)

# 2. 추가할 파일 리스트
new_files = [
    './data/league_of_legends/thresh.txt',
    './data/league_of_legends/senna.txt',
    './data/league_of_legends/lucian.txt'
]

# 3. 반복문으로 한꺼번에 로드하고 조각내기
all_new_chunks = []
for file_path in new_files:
    loader = TextLoader(file_path)
    new_doc = loader.load()
    chunks = text_splitter.split_documents(new_doc)
    all_new_chunks.extend(chunks) # 조각들을 바구니에 다 모읍니다.

# 4. 금고에 합치고 저장
vectorstore.add_documents(all_new_chunks)
vectorstore.save_local("my_lol_db")

print(f"성공! 총 {len(new_files)}개의 챔피언 지식이 추가되었습니다.")

성공! 총 3개의 챔피언 지식이 추가되었습니다.


In [107]:
### 디비 업데이트 버튼 추가

import gradio as gr
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
import os

# --- [1. 헬퍼 함수: 이름 추출 로직을 하나로 통일] ---
def extract_champion_info(docs_data):
    names = sorted(list(set([doc.page_content.split('\n')[0].strip() for doc in docs_data])))
    return names, ", ".join(names)

# --- [2. 프로그램 시작 시 초기 로드 로직] ---
# ⭐ 이 부분이 누락되면 처음 켤 때 champions_str이 뭔지 몰라서 에러가 나거나 옛날 데이터를 씁니다.
initial_loader = DirectoryLoader('./data/league_of_legends', glob="*.txt", loader_cls=TextLoader)
initial_docs = initial_loader.load()
champion_names, champions_str = extract_champion_info(initial_docs)

# --- [초기 설정 및 DB 로드] ---
global_vectorstore = FAISS.load_local("my_lol_db", embeddings_model, allow_dangerous_deserialization=True)
global_retriever = global_vectorstore.as_retriever(
    search_type='similarity', 
    search_kwargs={
        'k': 3,            # 최종 반환 문서 수
        'fetch_k': 10,     # 후보 문서 수
        'lambda_mult': 0.7 # 낮을수록 다양성 증가
    })

llm = ChatOpenAI(model_name="gpt-4.1-nano", temperature=0)
prompt = ChatPromptTemplate.from_messages([
    ("system", """너는 룬테라의 가이드야. 반드시 아래 규칙을 지켜서 답변해줘:
    [규칙]
    1. 제공된 [배경 지식] 정보를 바탕으로 친구에게 이야기하듯 친절한 반말로 답변해줘.
    2. 질문에 대한 정보가 [배경 지식]에 있다면 절대 숨기지 말고 상세히 알려줘.
    3. 만약 관련된 내용이 정말로 없다면 "미안해, 그 기록은 아직 우리 도서관에 없어."라고 답해줘."""),
    MessagesPlaceholder("chat_history"),
    ("system", "[배경 지식 정보]\n{context}"),
    ("human", "{user_input}")
])
rag_chain = prompt | llm | StrOutputParser()

# --- [핵심: HTML 생성 함수] ---
def get_description_html(champion_list_str):
    return f"""
    <div style="background-color: #f0f4f8; padding: 20px; border-radius: 15px; border-left: 5px solid #2196F3; margin-bottom: 20px;">
        <h3 style="margin-top: 0; color: #1a237e;">📜 룬테라의 소리에 귀를 기울여보세요</h3>
        <p style="font-size: 1.1em; color: #34495e;">반가워요! 나는 룬테라의 이야기를 들려주는 가이드입니다. 😊</p>
        <div style="background: white; padding: 10px; border-radius: 8px; font-weight: bold; color: #2196F3; display: inline-block;">
            📍 보관된 기록: {champion_list_str}
        </div>
    </div>
    """

# --- [DB 업데이트 함수] ---
def start_refresh():
    # 버튼 누르자마자 표시될 메시지
    return gr.update(value="🔄 지식 보관소를 정리하고 있습니다... 잠시만 기다려주세요."), gr.update(interactive=False)

def refresh_database():
    global global_vectorstore, global_retriever
    try:
        loader = DirectoryLoader('./data/league_of_legends', glob="*.txt", loader_cls=TextLoader)
        docs = loader.load()

        new_names, new_list_str = extract_champion_info(docs)

        chunks = text_splitter.split_documents(docs)
        global_vectorstore = FAISS.from_documents(chunks, embeddings_model)
        global_vectorstore.save_local("my_lol_db")
        global_retriever = global_vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': 5, 'fetch_k': 10, 'lambda_mult': 0.7})
        
        return (
            f"✅ 업데이트 완료! 현재 {len(new_names)}명의 기록이 있습니다.", 
            get_description_html(new_list_str), 
            gr.update(interactive=True)
        )
    except Exception as e:
        return f"❌ 오류 발생: {str(e)}", gr.update(), gr.update(interactive=True)

# --- [채팅 처리 함수] ---
def chat_with_bot(message, history):
    history_messages = []
    for msg in history:
        if msg['role'] == "user":
            history_messages.append(HumanMessage(content=msg['content']))
        elif msg['role'] == "assistant":
            history_messages.append(AIMessage(content=msg['content']))

    rel_docs = global_retriever.invoke(message)
    context_str = "\n".join([d.page_content for d in rel_docs])
    
    if not rel_docs or len(context_str.strip()) < 10:
        yield "미안해, 그 기록은 아직 우리 도서관에 보관되어 있지 않아. 😊"
        return

    partial_message = ""
    for chunk in rag_chain.stream({
        "chat_history": history_messages,
        "context": context_str,
        "user_input": message
    }):
        partial_message += chunk
        yield partial_message

# --- [화면 구성] ---
with gr.Blocks(theme="soft") as demo:
    gr.Markdown("# 🌟 LoL 유니버스 도서관")
    
    # 상단 HTML을 컴포넌트로 선언 (그래야 나중에 업데이트 가능)
    main_guide = gr.HTML(get_description_html(champions_str))
    
    with gr.Accordion("⚙️ 관리자 도구", open=False):
        update_status = gr.Markdown("파일을 추가한 후 아래 버튼을 누르면 지식이 갱신됩니다.")
        refresh_btn = gr.Button("📚 지식 보관소 새로고침", variant="primary")
    
    gr.ChatInterface(
        fn=chat_with_bot,
        examples=[[f"{champion_names[0]}에 대해 알려줘"], ["데마시아에 대해 알려줘"]]
    )
    
    # 1. 먼저 start_refresh 실행 (메시지 변경 & 버튼 비활성화)
    # 2. 이어서 refresh_database 실행 (실제 DB 작업 & 결과 반영)
    refresh_btn.click(
        fn=start_refresh, 
        outputs=[update_status, refresh_btn]
    ).then(
        fn=refresh_database, 
        outputs=[update_status, main_guide, refresh_btn]
    )

if __name__ == "__main__":
    demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7900


python(44436) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


* Running on public URL: https://5cd7a1111f22098e1f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [108]:
### 디비 업데이트 버튼 + 비밀번호


import gradio as gr
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
import os

# --- [1. 헬퍼 함수: 이름 추출 로직을 하나로 통일] ---
def extract_champion_info(docs_data):
    names = sorted(list(set([doc.page_content.split('\n')[0].strip() for doc in docs_data])))
    return names, ", ".join(names)

# --- [2. 프로그램 시작 시 초기 로드 로직] ---
# ⭐ 이 부분이 누락되면 처음 켤 때 champions_str이 뭔지 몰라서 에러가 나거나 옛날 데이터를 씁니다.
initial_loader = DirectoryLoader('./data/league_of_legends', glob="*.txt", loader_cls=TextLoader)
initial_docs = initial_loader.load()
champion_names, champions_str = extract_champion_info(initial_docs)

# --- [초기 설정 및 DB 로드] ---
global_vectorstore = FAISS.load_local("my_lol_db", embeddings_model, allow_dangerous_deserialization=True)
global_retriever = global_vectorstore.as_retriever(
    search_type='similarity', 
    search_kwargs={
        'k': 5,            # 최종 반환 문서 수
        'fetch_k': 10,     # 후보 문서 수
        'lambda_mult': 0.7 # 낮을수록 다양성 증가
    })

llm = ChatOpenAI(model_name="gpt-4.1-nano", temperature=0)
prompt = ChatPromptTemplate.from_messages([
    ("system", """너는 룬테라의 가이드야. 반드시 아래 규칙을 지켜서 답변해줘:
    [규칙]
    1. 제공된 [배경 지식] 정보를 바탕으로 친구에게 이야기하듯 친절한 반말로 답변해줘.
    2. 질문에 대한 정보가 [배경 지식]에 있다면 절대 숨기지 말고 상세히 알려줘.
    3. 만약 관련된 내용이 정말로 없다면 "미안해, 그 기록은 아직 우리 도서관에 없어."라고 답해줘."""),
    MessagesPlaceholder("chat_history"),
    ("system", "[배경 지식 정보]\n{context}"),
    ("human", "{user_input}")
])
rag_chain = prompt | llm | StrOutputParser()

# --- [핵심: HTML 생성 함수] ---
def get_description_html(champion_list_str):
    return f"""
    <div style="background-color: #f0f4f8; padding: 20px; border-radius: 15px; border-left: 5px solid #2196F3; margin-bottom: 20px;">
        <h3 style="margin-top: 0; color: #1a237e;">📜 룬테라의 소리에 귀를 기울여보세요</h3>
        <p style="font-size: 1.1em; color: #34495e;">반가워요! 나는 룬테라의 이야기를 들려주는 가이드입니다. 😊</p>
        <div style="background: white; padding: 10px; border-radius: 8px; font-weight: bold; color: #2196F3; display: inline-block;">
            📍 보관된 기록: {champion_list_str}
        </div>
    </div>
    """

# --- [DB 업데이트 함수] ---
def start_refresh():
    # 버튼 누르자마자 표시될 메시지
    return gr.update(value="🔄 지식 보관소를 정리하고 있습니다... 잠시만 기다려주세요."), gr.update(interactive=False)

def refresh_database():
    global global_vectorstore, global_retriever
    try:
        loader = DirectoryLoader('./data/league_of_legends', glob="*.txt", loader_cls=TextLoader)
        docs = loader.load()

        new_names, new_list_str = extract_champion_info(docs)

        chunks = text_splitter.split_documents(docs)
        global_vectorstore = FAISS.from_documents(chunks, embeddings_model)
        global_vectorstore.save_local("my_lol_db")
        global_retriever = global_vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': 5, 'fetch_k': 10, 'lambda_mult': 0.7})
        
        return (
            f"✅ 업데이트 완료! 현재 {len(new_names)}명의 기록이 있습니다.", 
            get_description_html(new_list_str), 
            gr.update(interactive=True)
        )
    except Exception as e:
        return f"❌ 오류 발생: {str(e)}", gr.update(), gr.update(interactive=True)

# --- [채팅 처리 함수] ---
def chat_with_bot(message, history):
    history_messages = []
    for msg in history:
        if msg['role'] == "user":
            history_messages.append(HumanMessage(content=msg['content']))
        elif msg['role'] == "assistant":
            history_messages.append(AIMessage(content=msg['content']))

    rel_docs = global_retriever.invoke(message)
    context_str = "\n".join([d.page_content for d in rel_docs])
    
    if not rel_docs or len(context_str.strip()) < 10:
        yield "미안해, 그 기록은 아직 우리 도서관에 보관되어 있지 않아. 😊"
        return

    partial_message = ""
    for chunk in rag_chain.stream({
        "chat_history": history_messages,
        "context": context_str,
        "user_input": message
    }):
        partial_message += chunk
        yield partial_message

# --- [화면 구성] ---
with gr.Blocks(theme="soft") as demo:
    gr.Markdown("# 🌟 LoL 유니버스 도서관")
    
    # 상단 HTML을 컴포넌트로 선언 (그래야 나중에 업데이트 가능)
    main_guide = gr.HTML(get_description_html(champions_str))
    
    with gr.Accordion("⚙️ 관리자 도구", open=False):
        admin_pw = gr.Textbox(label="관리자 암호", type="password")
        update_status = gr.Markdown("암호를 입력하고 버튼을 누르세요.")
        refresh_btn = gr.Button("📚 지식 보관소 새로고침", variant="primary")
    
    gr.ChatInterface(
        fn=chat_with_bot,
        examples=[[f"{champion_names[0]}에 대해 알려줘"], ["데마시아에 대해 알려줘"]]
    )

    # 버튼 클릭 시 비밀번호 체크 로직을 함수에 추가
    def protected_refresh(password):
        if password != "1234": # 실제로는 환경변수 등에 저장
            return "❌ 암호가 틀렸습니다.", gr.update(), gr.update(interactive=True)
        return refresh_database()
    
    # 1. 먼저 start_refresh 실행 (메시지 변경 & 버튼 비활성화)
    # 2. 이어서 refresh_database 실행 (실제 DB 작업 & 결과 반영)
    refresh_btn.click(
        fn=start_refresh, 
        outputs=[update_status, refresh_btn]
    ).then(
        fn=protected_refresh,
        inputs=[admin_pw], 
        outputs=[update_status, main_guide, refresh_btn]
    )

if __name__ == "__main__":
    demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7901


python(44452) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


* Running on public URL: https://3903999cf35823dab6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
